In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import time
import warnings

import joblib
import numpy as np
import polars as pl
import tensorflow as tf

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', message='X does not have valid feature names')
warnings.filterwarnings('ignore', category=UserWarning, module='xgboost')

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

TARGET_COL = 'label'
N_REPEATS = 3
WARMUP_SIZE = 1000


In [ ]:
# Carga de dataset y preprocesado, igual que en los cuadernos de ataques para TON-IOT
path_df = '../../DATASETS/dataSets_Reducidos/ton_iot/datos_TON_IoT_redux.csv'

df = pl.read_csv(path_df).drop_nulls()

cols_to_drop = ['label', 'type', 'src_ip', 'dst_ip']

X_df = df.drop(cols_to_drop).to_pandas()
y_np = df[TARGET_COL].to_numpy().astype(np.int8)

indices = np.arange(len(X_df))

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=SEED,
    stratify=y_np,
)

train_idx, val_idx = train_test_split(
    train_full_idx,
    test_size=0.20,
    random_state=SEED,
    stratify=y_np[train_full_idx],
)

X_train_full_df = X_df.iloc[train_full_idx].copy()
X_test_df = X_df.iloc[test_idx].copy()
X_train_df = X_df.iloc[train_idx].copy()

categorical_cols = ['proto', 'conn_state']
numeric_cols = [col for col in X_df.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ]
)

preprocessor.fit(X_train_df)

X_full_train_np = preprocessor.transform(X_train_full_df).astype(np.float32)
X_test_np = preprocessor.transform(X_test_df).astype(np.float32)
X_full_base = preprocessor.transform(X_df).astype(np.float32)

mlp_scaler = StandardScaler()
mlp_scaler.fit(X_full_train_np)
X_full_scaled_mlp = mlp_scaler.transform(X_full_base).astype(np.float32)

cnn_scaler = MinMaxScaler()
cnn_scaler.fit(X_full_train_np)
X_full_scaled_cnn = cnn_scaler.transform(X_full_base).astype(np.float32)
X_full_cnn = X_full_scaled_cnn.reshape(X_full_scaled_cnn.shape[0], X_full_scaled_cnn.shape[1], 1)

print(f'Train full: {len(X_full_train_np):,} muestras')
print(f'Test:       {len(X_test_np):,} muestras')
print(f'Total:      {len(X_full_base):,} muestras')
print(f'Features:   {X_full_base.shape[1]}')


In [ ]:
# RF

rf_path = '../model/ton-iot/rf_ton_iot.joblib'

rf_model = joblib.load(rf_path)
print('Modelo cargado')

# Warm-up
_ = rf_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_rf = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = rf_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_rf.append(t1 - t0)

tiempo_total_rf = float(np.mean(tiempos_rf))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_rf]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_rf:.4f} s')


In [ ]:
# XGBOOST

xgb_path = '../model/ton-iot/xgb_ton_iot.joblib'

xgb_model = joblib.load(xgb_path)
print('Modelo cargado')

# Warm-up
_ = xgb_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_xgb = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = xgb_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_xgb.append(t1 - t0)

tiempo_total_xgb = float(np.mean(tiempos_xgb))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_xgb]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_xgb:.4f} s')


In [ ]:
# LIGHTGBM

lgbm_path = '../model/ton-iot/lgbm_ton_iot.joblib'

lgbm_model = joblib.load(lgbm_path)
print('Modelo cargado')

# Warm-up
_ = lgbm_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_lgbm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = lgbm_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_lgbm.append(t1 - t0)

tiempo_total_lgbm = float(np.mean(tiempos_lgbm))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_lgbm]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_lgbm:.4f} s')


In [ ]:
# CATBOOST

catboost_path = '../model/ton-iot/catboost_ton_iot.joblib'

catboost_model = joblib.load(catboost_path)
print('Modelo cargado')

# Warm-up
_ = catboost_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_catboost = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = catboost_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_catboost.append(t1 - t0)

tiempo_total_catboost = float(np.mean(tiempos_catboost))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_catboost]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_catboost:.4f} s')


In [ ]:
# SVM

svm_path = '../model/ton-iot/svm_ton_iot.joblib'

svm_model = joblib.load(svm_path)
print('Modelo cargado')

# Warm-up
_ = svm_model.predict(X_full_base[:min(WARMUP_SIZE, len(X_full_base))])

# Medida latencia
tiempos_svm = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = svm_model.predict(X_full_base)
    t1 = time.perf_counter()
    tiempos_svm.append(t1 - t0)

tiempo_total_svm = float(np.mean(tiempos_svm))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_svm]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_svm:.4f} s')


In [ ]:
# MLP

mlp_path = '../model/ton-iot/mlp_ton_iot.joblib'

mlp_model = joblib.load(mlp_path)
print('Modelo cargado')

# Warm-up
_ = mlp_model.predict(X_full_scaled_mlp[:min(WARMUP_SIZE, len(X_full_scaled_mlp))], batch_size=4096, verbose=0)

# Medida latencia
tiempos_mlp = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = mlp_model.predict(X_full_scaled_mlp, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_mlp.append(t1 - t0)

tiempo_total_mlp = float(np.mean(tiempos_mlp))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_mlp]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_mlp:.4f} s')


In [ ]:
# CNN

cnn_path = '../model/ton-iot/cnn_ton_iot.joblib'

cnn_model = joblib.load(cnn_path)
print('Modelo cargado')

# Warm-up
_ = cnn_model.predict(X_full_cnn[:min(WARMUP_SIZE, len(X_full_cnn))], batch_size=4096, verbose=0)

# Medida latencia
tiempos_cnn = []

for _ in range(N_REPEATS):
    t0 = time.perf_counter()
    y_pred = cnn_model.predict(X_full_cnn, batch_size=4096, verbose=0)
    t1 = time.perf_counter()
    tiempos_cnn.append(t1 - t0)

tiempo_total_cnn = float(np.mean(tiempos_cnn))

print(f'Tiempos medidos: {[round(t, 4) for t in tiempos_cnn]}')
print(f'Tiempo medio de inferencia sobre todo el dataset: {tiempo_total_cnn:.4f} s')
